<a href="https://colab.research.google.com/github/aymensrihi/deep-learning-projects/blob/main/Untested.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Swin-Adaptive-SPSD: Swin Transformer with Stage-wise Adaptive Progressive Soft Pseudo-Labeling
✅ NEW: Dynamic stage selection based on model performance trends
✅ NEW: Performance-aware distillation focusing
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import einsum
from torchvision import transforms
import numpy as np
import random
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from tqdm import tqdm
import time
from collections import deque

# ============================================================================
# SWIN TRANSFORMER COMPONENTS (unchanged)
# ============================================================================

class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement

    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1, 2))


class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return self.net(x)


def create_mask(window_size, displacement, upper_lower, left_right):
    mask = torch.zeros(window_size ** 2, window_size ** 2)

    if upper_lower:
        mask[-displacement * window_size:, :-displacement * window_size] = float('-inf')
        mask[:-displacement * window_size, -displacement * window_size:] = float('-inf')

    if left_right:
        mask = rearrange(mask, '(h1 w1) (h2 w2) -> h1 w1 h2 w2', h1=window_size, h2=window_size)
        mask[:, -displacement:, :, :-displacement] = float('-inf')
        mask[:, :-displacement, :, -displacement:] = float('-inf')
        mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')

    return mask


def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x, y] for x in range(window_size) for y in range(window_size)]))
    distances = indices[None, :, :] - indices[:, None, :]
    return distances


class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        inner_dim = head_dim * heads

        self.heads = heads
        self.scale = head_dim ** -0.5
        self.window_size = window_size
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted = shifted

        if self.shifted:
            displacement = window_size // 2
            self.cyclic_shift = CyclicShift(-displacement)
            self.cyclic_back_shift = CyclicShift(displacement)
            self.upper_lower_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                             upper_lower=True, left_right=False), requires_grad=False)
            self.left_right_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                            upper_lower=False, left_right=True), requires_grad=False)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)

        if self.relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size) + window_size - 1
            self.pos_embedding = nn.Parameter(torch.randn(2 * window_size - 1, 2 * window_size - 1))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size ** 2, window_size ** 2))

        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        if self.shifted:
            x = self.cyclic_shift(x)

        b, n_h, n_w, _, h = *x.shape, self.heads

        qkv = self.to_qkv(x).chunk(3, dim=-1)
        nw_h = n_h // self.window_size
        nw_w = n_w // self.window_size

        q, k, v = map(
            lambda t: rearrange(t, 'b (nw_h w_h) (nw_w w_w) (h d) -> b h (nw_h nw_w) (w_h w_w) d',
                                h=h, w_h=self.window_size, w_w=self.window_size), qkv)

        dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale

        if self.relative_pos_embedding:
            dots += self.pos_embedding[self.relative_indices[:, :, 0], self.relative_indices[:, :, 1]]
        else:
            dots += self.pos_embedding

        if self.shifted:
            dots[:, :, -nw_w:] += self.upper_lower_mask
            dots[:, :, nw_w - 1::nw_w] += self.left_right_mask

        attn = dots.softmax(dim=-1)

        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        out = rearrange(out, 'b h (nw_h nw_w) (w_h w_w) d -> b (nw_h w_h) (nw_w w_w) (h d)',
                        h=h, w_h=self.window_size, w_w=self.window_size, nw_h=nw_h, nw_w=nw_w)
        out = self.to_out(out)

        if self.shifted:
            out = self.cyclic_back_shift(out)
        return out


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        self.attention_block = Residual(PreNorm(dim, WindowAttention(dim=dim,
                                                                     heads=heads,
                                                                     head_dim=head_dim,
                                                                     shifted=shifted,
                                                                     window_size=window_size,
                                                                     relative_pos_embedding=relative_pos_embedding)))
        self.mlp_block = Residual(PreNorm(dim, FeedForward(dim=dim, hidden_dim=mlp_dim)))

    def forward(self, x):
        x = self.attention_block(x)
        x = self.mlp_block(x)
        return x


class PatchMerging(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.downscaling_factor = downscaling_factor
        self.patch_merge = nn.Unfold(kernel_size=downscaling_factor, stride=downscaling_factor, padding=0)
        self.linear = nn.Linear(in_channels * downscaling_factor ** 2, out_channels)

    def forward(self, x):
        b, c, h, w = x.shape
        new_h, new_w = h // self.downscaling_factor, w // self.downscaling_factor
        x = self.patch_merge(x).view(b, -1, new_h, new_w).permute(0, 2, 3, 1)
        x = self.linear(x)
        return x


class StageModule(nn.Module):
    def __init__(self, in_channels, hidden_dimension, layers, downscaling_factor, num_heads, head_dim, window_size,
                 relative_pos_embedding):
        super().__init__()
        assert layers % 2 == 0, 'Stage layers need to be divisible by 2 for regular and shifted block.'

        self.patch_partition = PatchMerging(in_channels=in_channels, out_channels=hidden_dimension,
                                            downscaling_factor=downscaling_factor)

        self.layers = nn.ModuleList([])
        for _ in range(layers // 2):
            self.layers.append(nn.ModuleList([
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=False, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=True, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
            ]))

    def forward(self, x):
        x = self.patch_partition(x)
        for regular_block, shifted_block in self.layers:
            x = regular_block(x)
            x = shifted_block(x)
        return x.permute(0, 3, 1, 2)


class SwinTransformer_SPSD(nn.Module):
    def __init__(self, *, hidden_dim, layers, heads, channels=3, num_classes=1000, head_dim=32, window_size=7,
                 downscaling_factors=(4, 2, 2, 2), relative_pos_embedding=True):
        super().__init__()

        self.stage1 = StageModule(in_channels=channels, hidden_dimension=hidden_dim, layers=layers[0],
                                  downscaling_factor=downscaling_factors[0], num_heads=heads[0], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage2 = StageModule(in_channels=hidden_dim, hidden_dimension=hidden_dim * 2, layers=layers[1],
                                  downscaling_factor=downscaling_factors[1], num_heads=heads[1], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage3 = StageModule(in_channels=hidden_dim * 2, hidden_dimension=hidden_dim * 4, layers=layers[2],
                                  downscaling_factor=downscaling_factors[2], num_heads=heads[2], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage4 = StageModule(in_channels=hidden_dim * 4, hidden_dimension=hidden_dim * 8, layers=layers[3],
                                  downscaling_factor=downscaling_factors[3], num_heads=heads[3], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.LayerNorm(hidden_dim * (2 ** i)),
                nn.Linear(hidden_dim * (2 ** i), num_classes)
            )
            for i in range(4)
        ])

    def forward(self, img):
        stage_outputs = []
        x = self.stage1(img)
        stage_outputs.append(x)
        x = self.stage2(x)
        stage_outputs.append(x)
        x = self.stage3(x)
        stage_outputs.append(x)
        x = self.stage4(x)
        stage_outputs.append(x)

        predictions = [head(feat) for feat, head in zip(stage_outputs, self.heads)]
        return predictions


def swin_t_spsd(hidden_dim=96, layers=(2, 2, 6, 2), heads=(3, 6, 12, 24), **kwargs):
    return SwinTransformer_SPSD(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)


# ============================================================================
# DATASET CLASS
# ============================================================================

class DRDataset(Dataset):
    def __init__(self, root, domain_name, transform=None):
        self.root = os.path.join(root, domain_name)
        self.transform = transform
        self.classes = ['0', '1', '2', '3', '4']
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.samples = []
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')

        for class_name in self.classes:
            class_dir = os.path.join(self.root, class_name)
            if not os.path.exists(class_dir):
                continue

            class_idx = self.class_to_idx[class_name]
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(image_extensions):
                    self.samples.append((os.path.join(class_dir, img_name), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, target
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return self.__getitem__(random.randint(0, len(self) - 1))


# ============================================================================
# ✅ NEW: PERFORMANCE TRACKER FOR ADAPTIVE STAGE SELECTION
# ============================================================================

class PerformanceTracker:
    """
    Tracks model performance to determine which stages need distillation focus.

    Strategy:
    - Low accuracy (< threshold): Focus on early stages (1-2) → stabilize low-level features
    - Improving accuracy: Focus on late stages (3-4) → refine high-level representations
    - Maintains rolling window for smooth transitions
    """
    def __init__(self, window_size=5, low_acc_threshold=0.4, improving_threshold=0.6):
        self.window_size = window_size
        self.low_acc_threshold = low_acc_threshold
        self.improving_threshold = improving_threshold

        # Track accuracy history
        self.accuracy_history = deque(maxlen=window_size)
        self.batch_correct = 0
        self.batch_total = 0

        # Stage selection statistics
        self.stage_selection_counts = {0: 0, 1: 0, 2: 0, 3: 0}
        self.performance_phases = []

    def update_batch_stats(self, predictions, targets):
        """Update running batch accuracy statistics"""
        with torch.no_grad():
            _, predicted = torch.max(predictions, 1)
            self.batch_correct += (predicted == targets).sum().item()
            self.batch_total += targets.size(0)

    def get_current_accuracy(self):
        """Get current rolling window accuracy"""
        if len(self.accuracy_history) == 0:
            return 0.0
        return sum(self.accuracy_history) / len(self.accuracy_history)

    def is_improving(self):
        """Check if model is improving (accuracy trend going up)"""
        if len(self.accuracy_history) < 3:
            return False
        recent = list(self.accuracy_history)[-3:]
        return recent[-1] > recent[0]

    def commit_epoch_accuracy(self):
        """Commit batch statistics to epoch accuracy and reset"""
        if self.batch_total > 0:
            epoch_acc = self.batch_correct / self.batch_total
            self.accuracy_history.append(epoch_acc)
            self.batch_correct = 0
            self.batch_total = 0
            return epoch_acc
        return 0.0

    def select_stage_adaptive(self):
        """
        Adaptively select which stage to apply RB loss based on performance.

        Returns:
            int: Stage index (0-3)
            str: Reason for selection
        """
        current_acc = self.get_current_accuracy()

        # Phase 1: Low accuracy → Focus on early stages (foundation building)
        if current_acc < self.low_acc_threshold:
            stage_idx = random.choice([0, 1])  # Stages 1-2
            reason = f"LOW_ACC ({current_acc:.3f} < {self.low_acc_threshold})"
            phase = "early_stabilization"

        # Phase 2: Improving but not yet high → Balanced
        elif current_acc < self.improving_threshold:
            if self.is_improving():
                stage_idx = random.choice([2, 3])  # Stages 3-4
                reason = f"IMPROVING ({current_acc:.3f}, trend ↑)"
                phase = "late_refinement"
            else:
                stage_idx = random.choice([1, 2])  # Middle stages
                reason = f"PLATEAU ({current_acc:.3f}, trend →)"
                phase = "balanced"

        # Phase 3: High accuracy → Focus on late stages (fine-tuning)
        else:
            stage_idx = random.choice([2, 3])  # Stages 3-4
            reason = f"HIGH_ACC ({current_acc:.3f} ≥ {self.improving_threshold})"
            phase = "late_refinement"

        self.stage_selection_counts[stage_idx] += 1
        self.performance_phases.append(phase)

        return stage_idx, reason

    def get_statistics(self):
        """Get comprehensive statistics for logging"""
        total_selections = sum(self.stage_selection_counts.values())
        stage_distribution = {
            k: (v / total_selections * 100) if total_selections > 0 else 0
            for k, v in self.stage_selection_counts.items()
        }

        phase_counts = {}
        for phase in set(self.performance_phases):
            phase_counts[phase] = self.performance_phases.count(phase)

        return {
            'current_accuracy': self.get_current_accuracy(),
            'accuracy_history': list(self.accuracy_history),
            'stage_distribution': stage_distribution,
            'phase_counts': phase_counts,
            'is_improving': self.is_improving()
        }


# ============================================================================
# ✅ NEW: ADAPTIVE SWIN-SPSD WITH PERFORMANCE-BASED STAGE SELECTION
# ============================================================================

class AdaptiveSwinSPSD(nn.Module):
    """
    Adaptive Swin-SPSD with performance-aware stage selection.

    Key improvements:
    1. Tracks model performance over rolling window
    2. Dynamically selects stages for RB loss based on accuracy trends
    3. Early stages when learning, late stages when refining
    """
    def __init__(self, num_classes=5, hparams=None):
        super().__init__()
        if hparams is None:
            hparams = default_hparams()

        self.hparams = hparams
        self.lambda_ = hparams['RB_loss_weight']
        self.beta_T = hparams['alpha_T']
        self.n_steps = hparams['n_steps']
        self.step_count = 0
        self.n_classes = num_classes

        # ✅ NEW: Performance tracker for adaptive stage selection
        self.performance_tracker = PerformanceTracker(
            window_size=hparams.get('perf_window_size', 5),
            low_acc_threshold=hparams.get('low_acc_threshold', 0.4),
            improving_threshold=hparams.get('improving_threshold', 0.6)
        )

        self.network = swin_t_spsd(
            hidden_dim=96,
            layers=(2, 2, 6, 2),
            heads=(3, 6, 12, 24),
            num_classes=num_classes,
            channels=3,
            window_size=7
        )

        self.optimizer = torch.optim.AdamW(
            self.network.parameters(),
            lr=hparams["lr"],
            weight_decay=hparams['weight_decay']
        )

    def update(self, x, y):
        """✅ Training step with adaptive stage selection"""
        # Progressive soft pseudo-labeling weight
        beta_t = self.beta_T * ((self.step_count + 1) / self.n_steps)
        beta_t = max(0.0, min(beta_t, self.beta_T))
        self.step_count += 1

        # Forward pass
        outputs = self.network(x)
        z = outputs[-1]  # Final stage output

        # ✅ NEW: Adaptive stage selection based on performance
        stage_idx, selection_reason = self.performance_tracker.select_stage_adaptive()
        z_j = outputs[stage_idx]

        # One-hot encoding
        y_one_hot = torch.zeros(y.size(0), self.n_classes, device=x.device)
        y_one_hot.scatter_(1, y.unsqueeze(1), 1)

        # Soft pseudo-labels
        p = F.softmax(z, dim=1)
        p_j = F.softmax(z_j, dim=1)

        soft_p = beta_t * p + (1 - beta_t) * y_one_hot
        soft_p_j = beta_t * p_j + (1 - beta_t) * y_one_hot

        # Loss computation
        base_loss = F.cross_entropy(z, y)
        rb_loss = F.kl_div(
            torch.log(soft_p_j + 1e-10),
            soft_p.detach(),
            reduction='batchmean'
        )

        loss = base_loss + self.lambda_ * rb_loss

        # Optimization
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # ✅ NEW: Update performance tracker
        self.performance_tracker.update_batch_stats(z, y)

        return {
            'loss': loss.item(),
            'base_loss': base_loss.item(),
            'rb_loss': rb_loss.item(),
            'beta_t': beta_t,
            'selected_stage': stage_idx,
            'selection_reason': selection_reason
        }

    def commit_epoch_stats(self):
        """Commit batch statistics to epoch-level tracking"""
        return self.performance_tracker.commit_epoch_accuracy()

    def get_performance_stats(self):
        """Get comprehensive performance statistics"""
        return self.performance_tracker.get_statistics()

    def predict(self, x):
        outputs = self.network(x)
        return outputs[-1]


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def default_hparams():
    return {
        'data_augmentation': True,
        'RB_loss_weight': 0.7,
        'alpha_T': 0.8,
        'n_steps': None,
        'lr': 5e-5,
        'weight_decay': 0.05,
        'batch_size': 32,
        # ✅ NEW: Adaptive stage selection parameters
        'perf_window_size': 5,        # Rolling window for accuracy tracking
        'low_acc_threshold': 0.4,     # Threshold for "low" accuracy
        'improving_threshold': 0.6    # Threshold for "improving" accuracy
    }


def get_transforms(augment=True):
    transform_list = [transforms.Resize((224, 224))]
    if augment:
        transform_list.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.4, contrast=0.4)
        ])
    transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transforms.Compose(transform_list)


@torch.no_grad()
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        predictions = model.predict(x)
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    return correct / total if total > 0 else 0.0


def format_time(seconds):
    """Format seconds into readable time string"""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        mins = seconds / 60
        return f"{mins:.1f}m"
    else:
        hours = seconds / 3600
        return f"{hours:.1f}h"


def train_multi_source_dg(data_root, test_domain, num_epochs=10, hparams=None):
    """
    ✅ NEW: Training with Adaptive Stage Selection
    """
    if hparams is None:
        hparams = default_hparams()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    all_domains = ['aptos', 'eyepacs', 'messidor', 'messidor_2']
    train_domains = [d for d in all_domains if d != test_domain]

    print(f"Training domains: {train_domains}")
    print(f"Test domain: {test_domain}\n")

    train_transform = get_transforms(augment=True)
    test_transform = get_transforms(augment=False)

    train_datasets = [DRDataset(data_root, d, train_transform) for d in train_domains]
    test_dataset = DRDataset(data_root, test_domain, test_transform)

    train_loaders = [
        DataLoader(ds, batch_size=hparams['batch_size'], shuffle=True, num_workers=2)
        for ds in train_datasets
    ]
    test_loader = DataLoader(test_dataset, batch_size=hparams['batch_size'],
                            shuffle=False, num_workers=2)

    print(f"Training samples: {sum(len(ds) for ds in train_datasets)}")
    print(f"Test samples: {len(test_dataset)}\n")

    steps_per_epoch = max(len(loader) for loader in train_loaders)
    total_steps = steps_per_epoch * num_epochs
    hparams['n_steps'] = total_steps

    print("=" * 80)
    print("✅ ADAPTIVE SWIN-SPSD CONFIGURATION:")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Total epochs: {num_epochs}")
    print(f"  Total training steps: {total_steps}")
    print(f"  β progression: 0.0 → {hparams['alpha_T']}")
    print(f"  Performance window: {hparams['perf_window_size']} epochs")
    print(f"  Low accuracy threshold: {hparams['low_acc_threshold']}")
    print(f"  Improving threshold: {hparams['improving_threshold']}")
    print("=" * 80)
    print()

    model = AdaptiveSwinSPSD(num_classes=5, hparams=hparams).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}\n")

    best_acc = 0.0
    training_start_time = time.time()

    # ✅ NEW: Track stage selection over training
    stage_history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_base_loss = 0
        epoch_rb_loss = 0
        n_batches = 0

        epoch_start_time = time.time()
        epoch_stage_selections = []

        train_iters = [iter(loader) for loader in train_loaders]
        max_batches = max(len(loader) for loader in train_loaders)

        epochs_left = num_epochs - epoch - 1
        pbar = tqdm(
            range(max_batches),
            desc=f"Epoch {epoch+1}/{num_epochs} ({epochs_left} left)",
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
        )

        for batch_idx in pbar:
            all_x, all_y = [], []
            for train_iter, loader in zip(train_iters, train_loaders):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(loader)
                    x, y = next(train_iter)
                all_x.append(x)
                all_y.append(y)

            x = torch.cat(all_x).to(device)
            y = torch.cat(all_y).to(device)

            step_vals = model.update(x, y)

            epoch_loss += step_vals['loss']
            epoch_base_loss += step_vals['base_loss']
            epoch_rb_loss += step_vals['rb_loss']
            n_batches += 1

            epoch_stage_selections.append(step_vals['selected_stage'])

            # Progress tracking
            progress_pct = ((epoch * max_batches + batch_idx + 1) / (num_epochs * max_batches)) * 100

            # ✅ NEW: Show adaptive stage selection info
            pbar.set_postfix({
                'loss': f"{step_vals['loss']:.4f}",
                'β_t': f"{step_vals['beta_t']:.4f}",
                'stage': f"S{step_vals['selected_stage']+1}",
                'progress': f"{progress_pct:.1f}%"
            })

        # ✅ NEW: Commit epoch statistics
        epoch_train_acc = model.commit_epoch_stats()
        stage_history.append(epoch_stage_selections)

        epoch_time = time.time() - epoch_start_time

        # Evaluation
        test_acc = evaluate(model, test_loader, device)

        # ✅ NEW: Get performance statistics
        perf_stats = model.get_performance_stats()

        # Enhanced epoch summary
        elapsed_total = time.time() - training_start_time
        avg_epoch_time = elapsed_total / (epoch + 1)
        eta = avg_epoch_time * epochs_left

        print(f"\n{'='*80}")
        print(f"EPOCH {epoch+1}/{num_epochs} COMPLETE | {epochs_left} epochs remaining")
        print(f"{'='*80}")
        print(f"  Time: {format_time(epoch_time)} | Avg: {format_time(avg_epoch_time)} | ETA: {format_time(eta)}")
        print(f"  Loss: {epoch_loss/n_batches:.4f}")
        print(f"  Base Loss: {epoch_base_loss/n_batches:.4f}")
        print(f"  RB Loss: {epoch_rb_loss/n_batches:.4f}")
        print(f"  Training Accuracy: {epoch_train_acc*100:.2f}%")
        print(f"  Test Accuracy ({test_domain}): {test_acc*100:.2f}%")

        # ✅ NEW: Adaptive stage selection summary
        print(f"\n  📊 Adaptive Stage Selection:")
        print(f"     Rolling Accuracy: {perf_stats['current_accuracy']*100:.2f}%")
        print(f"     Trend: {'↑ Improving' if perf_stats['is_improving'] else '→ Stable'}")
        print(f"     Stage Distribution: " +
              ", ".join([f"S{k+1}={v:.1f}%" for k, v in sorted(perf_stats['stage_distribution'].items())]))

        if perf_stats['phase_counts']:
            print(f"     Phases: " +
                  ", ".join([f"{k}={v}" for k, v in perf_stats['phase_counts'].items()]))

        if test_acc > best_acc:
            best_acc = test_acc
            improvement = (test_acc - best_acc) * 100 if epoch > 0 else test_acc * 100
            print(f"\n  ✓ New best accuracy! (↑ {improvement:.2f}%)")
        print(f"{'='*80}\n")

    total_training_time = time.time() - training_start_time

    # ✅ NEW: Final adaptive selection analysis
    print("\n" + "=" * 80)
    print("✅ TRAINING COMPLETE - ADAPTIVE SELECTION ANALYSIS:")
    print(f"  Total training time: {format_time(total_training_time)}")
    print(f"  Best test accuracy: {best_acc*100:.2f}%")

    final_stats = model.get_performance_stats()
    print(f"\n  Final Stage Distribution:")
    for stage_idx, pct in sorted(final_stats['stage_distribution'].items()):
        print(f"    Stage {stage_idx+1}: {pct:.1f}%")

    print(f"\n  Accuracy Progression:")
    for i, acc in enumerate(final_stats['accuracy_history'], 1):
        print(f"    Epoch {i}: {acc*100:.2f}%")

    print("=" * 80)

    return model, best_acc, stage_history


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    try:
        import einops
    except ImportError:
        print("Installing einops...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'einops', '-q'])
        print("✓ einops installed\n")

    print("=" * 80)
    print("Adaptive Swin-SPSD: Performance-Aware Domain Generalization")
    print("✅ NEW: Dynamic stage selection based on accuracy trends")
    print("✅ NEW: Early stages for low accuracy, late stages for refinement")
    print("=" * 80)

    data_path = "/content/DR"

    if not os.path.exists(data_path):
        print(f"\nERROR: Dataset not found at {data_path}")
        print("Please update the data_path variable")
        exit(1)

    print(f"\n✓ Dataset found at: {data_path}")
    print(f"✓ Domains: {sorted(os.listdir(data_path))}\n")

    test_domain = 'messidor_2'
    num_epochs = 10

    hparams = default_hparams()

    print(f"Configuration:")
    print(f"  Backbone: Swin-Tiny")
    print(f"  Test domain: {test_domain}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {hparams['batch_size']}")
    print(f"  Learning rate: {hparams['lr']}")
    print(f"  λ (RB weight): {hparams['RB_loss_weight']}")
    print(f"  β_T (max PSPL): {hparams['alpha_T']}")
    print(f"  Performance window: {hparams['perf_window_size']} epochs")
    print(f"  Low acc threshold: {hparams['low_acc_threshold']}")
    print(f"  Improving threshold: {hparams['improving_threshold']}\n")

    model, best_acc, stage_history = train_multi_source_dg(
        data_root=data_path,
        test_domain=test_domain,
        num_epochs=num_epochs,
        hparams=hparams
    )

    print("\n" + "=" * 80)
    print(f"✓ Training complete!")
    print(f"✓ Best accuracy on {test_domain}: {best_acc*100:.2f}%")
    print("=" * 80)

    print("\n" + "=" * 80)
    print("Testing on all domains:")
    print("=" * 80)

    all_domains = ['aptos', 'eyepacs', 'messidor', 'messidor_2']
    test_transform = get_transforms(augment=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    results = {}
    for domain in all_domains:
        test_dataset = DRDataset(data_path, domain, test_transform)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
        acc = evaluate(model, test_loader, device)
        results[domain] = acc
        marker = "✓" if domain != test_domain else "→"
        print(f"  {marker} {domain:15s}: {acc*100:.2f}%")

    avg_acc = sum(results.values()) / len(results)
    print(f"\n  Average accuracy: {avg_acc*100:.2f}%")
    print("=" * 80)